## Initial GridWorld setup

Source: `course_project_v2.pdf`, page 3. Required: a 10 × 10 grid, 10 hazards, 3 goals, one starting square, and a discount factor of 0.9.

The fixed square locations below are our design choice. The handout refers to Lecture 1 but does not specify what happens at goals or hazards. We retain the supplied notebook's rule pending verification against Lecture 1: from a goal or hazard, the next move returns to the start with probability 0.8; otherwise, normal movement applies. Reaching either type of square does not end a run.

Normal movement follows the chosen direction with probability 0.8 and splits the remaining probability equally across the other three directions. Movement beyond the board stays at the edge. Costs depend on the square being left: goal 0, ordinary square 0.2, hazard 1. These rules and costs are inherited choices, not explicit PDF requirements.

Run the cells in order. The table below is reproducible example experience, not a fixed training dataset. Learning methods should request fresh moves from `generative_sample`, rather than read the probability array directly. The short run settings are initial checks, not tuned experiment settings. Blackjack and the learning methods remain separate project work.


In [ ]:
import numpy as np
import numpy.linalg as la


In [ ]:
import numpy as np

"""
GridWorld setup: 10 x 10 squares, four actions (Up, Down, Left, Right).
P[next_state, current_state, action] stores movement probabilities.
Goals and hazards are not terminal in this model: on the following move,
return to the start with probability 1-q; otherwise move normally.
This behavior is inherited from the original notebook, not specified by the PDF.
Costs are charged for the current square before moving; lower is better.
"""

GRID_ROWS    = 10
GRID_COLS    = 10
N_STATES     = GRID_ROWS * GRID_COLS   # 100
N_ACTIONS    = 4                        # 0=Up  1=Down  2=Left  3=Right

GOAL_STATES  = [9, 90, 99]
HAZARD_STATES = [12, 17, 24, 36, 43, 58, 62, 75, 81, 87]
GAMMA = 0.9            # discount factor required by the handout
START_STATE  = 0        # initial state with probability 1
N_STEPS      = 20       # template visualization horizon (SPMD uses T)

_DELTAS = [(-1,0), (1,0), (0,-1), (0,1)]        # Up Down Left Right
_OTHER_ACTIONS   = {0:[1,2,3], 1:[0,2,3], 2:[0,1,3], 3:[0,1,2]}  # all other actions, including the opposite direction

def s2rc(s):      return s // GRID_COLS, s % GRID_COLS
def rc2s(r, c):   return r * GRID_COLS + c
def clip(r, c):   return max(0, min(r, GRID_ROWS-1)), max(0, min(c, GRID_COLS-1))

def build_default_P(p: float = 0.8, q: float = 0.2) -> np.ndarray:
    """p: intended-move probability; q: normal-move probability at special squares."""
    if not (0 <= p <= 1 and 0 <= q <= 1):
        raise ValueError("p and q must be between zero and one")
    P = np.zeros((N_STATES, N_STATES, N_ACTIONS))
    for s in range(N_STATES):
        p_mult = 1.
        if s in GOAL_STATES or s in HAZARD_STATES:
            P[START_STATE, s, :] = 1.-q       # return
            p_mult = q
        r, c = s2rc(s)
        for a, (dr, dc) in enumerate(_DELTAS):
            nr, nc = clip(r + dr, c + dc)
            P[rc2s(nr, nc), s, a] += p * p_mult
            for pa in _OTHER_ACTIONS[a]:
                pdr, pdc = _DELTAS[pa]
                pnr, pnc = clip(r + pdr, c + pdc)
                P[rc2s(pnr, pnc), s, a] += (1.-p) * p_mult/len(_OTHER_ACTIONS[a])
    return P


c_G = 0
c_R = 0.2
c_H = 1.0
def build_default_c() -> np.ndarray:
    c = c_R * np.ones((N_STATES, N_ACTIONS))
    for s in GOAL_STATES:
        c[s,:] = c_G
    c[HAZARD_STATES,:] = c_H

    return c

P  = build_default_P(p=0.8)
pi = np.ones((N_STATES, N_ACTIONS), dtype=float)/N_ACTIONS
c  = build_default_c()
N_STATES, N_ACTIONS = P.shape[1:]

def generative_sample(s,a,rng):
    """
    :input s: input state
    :input a: input action
    :input rng: random generator
    :output s': next state sampled from P(*|s,a)

    Note: Do not modify P, as it will destroy this function
    """
    assert 0 <= s < N_STATES, "illegal state %s" % s
    assert 0 <= a < N_ACTIONS, "illegal action %s" % a

    return rng.choice(N_STATES, p=P[:,s,a])


# Monte Carlo estimator configuration
T = 10             # Trajectory length 
n_rollouts = 10  # Number of sampled trajectories per state-action pair
n_eval_rollouts = 100  # Number of trajectories for starting-state evaluation
k = 10              # Number of SPMD iterations


In [ ]:
# Generate a small, repeatable example table without adding dependencies.
def sample_experience(n_steps=100, seed=692):
    rng = np.random.default_rng(seed)
    records = np.empty(n_steps, dtype=[
        ("step", int), ("state", int), ("action", int),
        ("next_state", int), ("cost", float)
    ])
    state = START_STATE
    for step in range(n_steps):
        action = int(rng.choice(N_ACTIONS, p=pi[state]))
        next_state = generative_sample(state, action, rng)
        records[step] = (step, state, action, next_state, c[state, action])
        state = next_state
    return records

initial_data = sample_experience()
print("Actions: 0=Up, 1=Down, 2=Left, 3=Right")
print("step | state | action | next_state | cost")
for row in initial_data[:15]:
    print(f"{row['step']:4d} | {row['state']:5d} | {row['action']:6d} | "
          f"{row['next_state']:10d} | {row['cost']:.1f}")


In [ ]:
import matplotlib.pyplot as plt
# Show the fixed board. Each square is labeled with its state number.
from matplotlib.colors import ListedColormap

board = np.zeros(N_STATES, dtype=int)
board[HAZARD_STATES] = 1
board[GOAL_STATES] = 2
board[START_STATE] = 3
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(board.reshape(GRID_ROWS, GRID_COLS),
          cmap=ListedColormap(["#eeeeee", "#ef9a9a", "#a5d6a7", "#90caf9"]),
          vmin=0, vmax=3)
for state in range(N_STATES):
    row, col = s2rc(state)
    label = "S" if state == START_STATE else (
        "G" if state in GOAL_STATES else "H" if state in HAZARD_STATES else "")
    ax.text(col, row, f"{state}\n{label}" if label else str(state),
            ha="center", va="center", fontsize=9)
ax.set_xticks(np.arange(-0.5, GRID_COLS, 1), minor=True)
ax.set_yticks(np.arange(-0.5, GRID_ROWS, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="both", bottom=False, left=False, labelbottom=False, labelleft=False)
ax.set_title("GridWorld: S = start, G = goal, H = hazard")
plt.show()
